Cell 1: Initialize Gemini & Define a Tool with @tool

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(override=True)

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

Cell 2: Bind the Tool to the Model (bind_tools)
bind_tools() attaches your function's name, parameter schema, and docstring to Gemini so the model knows it exists.

In [3]:
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_7515111586077841831': 'Eo4BCosBAWkUfRM6MZN8kKO82vr4zBoWK2HudreHbJynQoLvlk5BOo6XYv6C8tezYyxDHtZEloe5w+x5B7i8VxTug1J0TFowW1XJiEZ4ok8i5esAHen/OCE0rGILbnScnPdbygeX9Tror+o2wNR6v0Z4rZIcacRdXCEQLiGx4TmeHsYFyzKtW/AoKdnkkKXqOw=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'tool_call', 'model_provider': 'google_genai'} id='lc_run--01a20814-4acc-7036-b131-6cd974645031-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_7515111586077841831', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 54, 'output_tokens': 16, 'total_tokens': 70, 'input_token_details': {'cache_read': 0}}


Cell 3: Inspect response.tool_calls
Notice response.text is empty—Gemini does not run Python functions itself. Instead, it returns a structured tool_calls instruction telling you which function to run and with what arguments:

In [4]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


Cell 4: Complete Tool Execution Loop (Passing Tool Output Back to Gemini)
Krish then shows the 3-step manual tool-calling loop:

Step 1: Model generates a tool call.

Step 2: Your code executes the tool and wraps the output in a ToolMessage.

Step 3: You pass the conversation history (messages) back to Gemini so it generates the final natural-language answer

In [5]:
# Step 1: Model generates tool call
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute each requested tool call and append the ToolMessage
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Send the tool result back to the model for the final reply
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.
